# Benchmark Validation

Review 50 synthetic requirement/clause pairs, reproduce citation and provenance checks, and inspect the new three-class labels. Original team files remain unchanged. This notebook uses the Python standard library and Jupyter display support; no model or paid API is required.

Sources: [team spreadsheet](https://docs.google.com/spreadsheets/d/1uFHTLkBYmrktTkqfwUdFjSMYvDdwiHC61SiBqQt2r5w/edit), [NASA NPR 7150.2D](https://swehb.nasa.gov/spaces/SITE/pages/123601159/NPR%2B7150.2D), and [Data Card](../docs/benchmark-data-card.md).

In [1]:
from pathlib import Path
import csv
import hashlib
import html
import importlib.util
import json
import subprocess
from collections import Counter
from IPython.display import HTML, display

# Use the local checkout. In Colab, clone the new branch into a separate folder.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data/benchmark/manifest.json').exists()), None)
if ROOT is None:
    ROOT = Path.cwd() / 'compliance-benchmark-work'
    if not ROOT.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'alan/benchmark-validation-data-card',
            'https://github.com/Break-Through-Tech/Aerospace-1C-compliance-copilot.git',
            str(ROOT),
        ], check=True)
    if not (ROOT / 'data/benchmark/manifest.json').exists():
        raise FileNotFoundError('Open this notebook in the benchmark branch checkout.')

spec = importlib.util.spec_from_file_location('benchmark_validator', ROOT / 'scripts/validate_benchmark.py')
validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validator)
bundle = validator.load_bundle(ROOT)
rows, reviews = bundle['records'], bundle['review']
clauses = {r['swe_id']: r for r in bundle['clauses']}

def show_table(headers, values):
    head = ''.join('<th>' + html.escape(str(v)) + '</th>' for v in headers)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(v)) + '</td>' for v in row) + '</tr>' for row in values)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

print('Loaded', len(rows), 'examples and', len(clauses), 'complete clause references.')

Loaded 50 examples and 44 complete clause references.


## Verify the package

Checks cover all 50 citations, unique IDs, allowed labels, original-row linkage, required source lists/table, and file hashes. They do not replace semantic source review.

In [2]:
result = validator.validate(bundle)
print(json.dumps(result, indent=2))
assert result['citation_matches'] == 50
assert result['verdict_counts'] == {'Meets': 28, 'Partial': 10, 'Gap': 12}

{
  "status": "PASS",
  "row_count": 50,
  "unique_swe_ids": 44,
  "source_label_counts": {
    "Compliant": 28,
    "Gap": 22
  },
  "verdict_counts": {
    "Meets": 28,
    "Gap": 12,
    "Partial": 10
  },
  "revised_text_count": 14,
  "normalized_id_count": 20,
  "citation_matches": 50,
  "file_hashes_checked": true
}


## Label distribution

Meets covers all mandatory aspects. Partial covers some but omits others. Gap provides no material coverage or expressly allows a violation. For binary gap detection, both Partial and Gap are positive cases.

In [3]:
counts = Counter(r['verdict'] for r in rows)
show_table(['Verdict', 'Count', 'Share'], [
    [label, counts[label], f'{counts[label] / len(rows):.0%}']
    for label in ['Meets', 'Partial', 'Gap']
])
matrix = Counter((r['source_label'], r['final_verdict']) for r in reviews)
show_table(['Original label', 'Meets', 'Partial', 'Gap'], [
    [label, *[matrix[(label, verdict)] for verdict in ['Meets', 'Partial', 'Gap']]]
    for label in ['Compliant', 'Gap']
])

Verdict,Count,Share
Meets,28,56%
Partial,10,20%
Gap,12,24%


Original label,Meets,Partial,Gap
Compliant,28,0,0
Gap,0,10,12


## Trace changes

Fourteen examples clarify ambiguous scope or incomplete obligations. Twenty IDs receive consistent zero padding. The original text and original labels remain in `data/benchmark/source.json`.

In [4]:
changes = [r for r in reviews if r['text_changed'] == 'true']
show_table(['ID', 'Original wording review', 'Final verdict', 'Clarification'], [
    [r['requirement_id'], r['original_wording_review'], r['final_verdict'], r['review_note']]
    for r in changes
])
assert len(changes) == 14
assert sum(r['source_id'] != r['requirement_id'] for r in reviews) == 20

ID,Original wording review,Final verdict,Clarification
NASA-SR-003,Partial,Meets,"Covers every Class B relationship in Table 1, including hazards and non-conformances."
NASA-SR-004,Partial,Meets,Requires every listed requirements-management action and makes the technical-specification role explicit.
NASA-SR-009,Partial,Meets,Includes both phases and all four minimum analysis purposes in the clause.
NASA-SR-015,Partial,Meets,"Includes hazardous events, causes, and mitigation techniques, with verification through test."
NASA-SR-016,Partial,Meets,"Includes Agency policy, project requirements, and risks from third-party or reused software."
NASA-SR-021,Partial,Meets,"Covers all five artifact categories in items a through e, with both performance and reporting."
NASA-SR-024,Partial,Partial,Implements categories but leaves their severity criteria undefined; the clause requires clear levels.
NASA-SR-025,Meets,Meets,Explicitly covers all six conditions without naming the expected SWE ID inside the example.
NASA-SR-029,Partial,Meets,Explicitly assesses contributing processes and closes corrective actions for every high-severity case.
NASA-SR-035,Meets,Meets,"Plans, tracks, and ensures project-specific training across the stated project-personnel scope."


## Inspect clause evidence

The examples below show a complete match, an incomplete match, and an explicit violation. The full reference includes NASA lists and the traceability table, which the existing abbreviated source CSV does not retain.

In [5]:
by_id = {r['requirement_id']: r for r in rows}
for req_id in ['NASA-SR-009', 'NASA-SR-024', 'NASA-SR-046']:
    row = by_id[req_id]
    reference = clauses[row['swe_id']]
    display(HTML('<h3>' + req_id + ': ' + row['verdict'] + '</h3>' +
        '<p><b>Requirement:</b> ' + html.escape(row['requirement_text']) + '</p>' +
        '<p><b>NASA clause:</b> ' + html.escape(reference['clause_text']) + '</p>' +
        '<p><b>Review:</b> ' + html.escape(row['rationale']) + '</p>' +
        '<p><a href="' + html.escape(reference['pdf_url'], quote=True) + '">NASA PDF, page ' + str(reference['pdf_page']) + '</a></p>'))

## Ready for evaluation

Use `requirements.csv` as the benchmark and `clauses.json` for the standard reference. Keep verdicts, rationales, and gap types out of model inputs. Six SWE IDs have two examples each, so keep related clauses together when creating training/evaluation partitions.

This notebook confirms dataset checks, not model performance or real project compliance. Team review is pending. See the concise [validation notes](../docs/benchmark-validation.md) for the review decisions.